In [19]:
# Cell 1 - Loading the knowledge base
import pandas as pd
import json
from typing import Optional
print("Loading knowledge base...")

# Load the full knowledge base you already generated
kb_df = pd.DataFrame()

try:
    with open('knowledge_base_full.json', 'r') as f:
        kb_data = json.load(f)
    kb_df = pd.DataFrame(kb_data)
    print(f"✅ Loaded knowledge base with {len(kb_df)} nouns")
    print(f"✅ Properties per noun: {len(kb_df.columns) - 1}")  # -1 for 'noun' column
except FileNotFoundError:
    print("❌ knowledge_base_full.json not found!")
    print("Please make sure you've generated the knowledge base first (Cell 12)")

# Display summary
print(f"\nKnowledge Base Overview:")
print(f"Total nouns: {len(kb_df)}")
#print(f"\nFirst 5 nouns:")
#print(kb_df['noun'].head())

#print(f"\nSample data for '{kb_df['noun'].iloc[0]}':")
#print(kb_df.iloc[0])

Loading knowledge base...
✅ Loaded knowledge base with 1852 nouns
✅ Properties per noun: 25

Knowledge Base Overview:
Total nouns: 1852


In [20]:
# Cell 15 - Question Selection Algorithm

import numpy as np

class TwentyQuestionsGame:
    """
    20 Questions game using property-based filtering
    """

    def __init__(self, knowledge_base_df):
        """
        Initialize with the knowledge base
        """
        self.kb = knowledge_base_df.copy()
        self.all_nouns = set(self.kb['noun'].values)
        self.properties = [col for col in self.kb.columns if col != 'noun']

        # Separate binary and scale properties
        self.binary_props = [p for p in self.properties
                            if self.kb[p].dtype == bool or
                            set(self.kb[p].dropna().unique()).issubset({True, False, 1, 0})]
        self.scale_props = [p for p in self.properties if p not in self.binary_props]

        print(f"Game initialized with {len(self.all_nouns)} nouns")
        print(f"Binary properties: {len(self.binary_props)}")
        print(f"Scale properties: {len(self.scale_props)}")

    def calculate_information_gain(self, remaining_nouns, property_name, threshold=None):
        """
        Calculate how well a question splits the remaining nouns
        Higher score = better question
        """
        if len(remaining_nouns) <= 1:
            return 0

        subset = self.kb[self.kb['noun'].isin(remaining_nouns)]
        total = len(subset)


        if property_name in self.binary_props:
            # Binary question: how evenly does it split?
            true_count = subset[property_name].sum()
            false_count = len(subset) - true_count

            # Best split is 50/50
            worst_case_remaining = max(true_count, false_count)
            elimination_power = total - worst_case_remaining

            # tie-breaker: smaller absolute difference preferred
            ratio = abs(true_count - false_count)
            return elimination_power, ratio

        elif property_name in self.scale_props and threshold is not None:
            # Scale question with threshold
            above = (subset[property_name] > threshold).sum()
            below = total - above

            ratio = abs(above - below)
            worst_case_remaining = max(above, below)
            elimination_power = total - worst_case_remaining

            return elimination_power, ratio

        return 0, 0

    def find_best_question(self, remaining_nouns, asked_questions=None):
        """
        Find the question that best splits the remaining nouns and uses elimination power
        """
        if len(remaining_nouns) <= 1:
            return None, None, None

        # Track 
        if asked_questions is None:
            asked_questions = set()

        best_elim = -1
        best_tiebreak_metric = float('inf')
        best_property = None
        best_threshold = None
        best_question_text = None

        subset = self.kb[self.kb['noun'].isin(remaining_nouns)]
        total = len(subset)
        
        # Try binary properties
        for prop in self.binary_props:
            # If already asked
            if (prop, None) in asked_questions:
                continue
            elimination_power, tiebreak_metric = self.calculate_information_gain(remaining_nouns, prop)
            if (elimination_power > best_elim) or (elimination_power == best_elim and tiebreak_metric < best_tiebreak_metric):
                best_elim = elimination_power
                best_tiebreak_metric = tiebreak_metric
                best_property = prop
                best_threshold = None
                best_question_text = self._format_binary_question(prop)

        # Try scale properties with different thresholds
        for prop in self.scale_props:
            values = subset[prop].dropna()

            if len(values) == 0:
                continue
                
            # Try thresholds
            thresholds = [values.median(), values.quantile(0.25), values.quantile(0.75)]
            for threshold in thresholds:
                # If already asked
                if (prop, None) in asked_questions:
                    continue
                elimination_power, tiebreak_metric = self.calculate_information_gain(remaining_nouns, prop, threshold)
                if (elimination_power > best_elim) or (elimination_power == best_elim and tiebreak_metric < best_tiebreak_metric):
                    best_elim = elimination_power
                    best_tiebreak_metric = tiebreak_metric
                    
                    best_property = prop
                    best_threshold = threshold
                    best_question_text = self._format_scale_question(prop, threshold)

                    
        return best_property, best_threshold, best_question_text

    def _format_binary_question(self, property_name):
        """Format a binary property as a yes/no question"""
        questions = {
            'is_living': "Is it a living thing?",
            'is_animal': "Is it an animal?",
            'is_plant': "Is it a plant?",
            'is_object': "Is it a physical object?",
            'is_abstract': "Is it an abstract concept?",
            'is_natural': "Is it found in nature?",
            'is_artificial': "Is it man-made?",
            'can_fly': "Can it fly?",
            'lives_in_water': "Does it live in water?",
            'is_predator': "Is it a predator?",
            'requires_electricity': "Does it require electricity?",
            'is_food': "Is it food?",
            'is_vehicle': "Is it a vehicle?",
            'is_tool': "Is it a tool?",
            'is_furniture': "Is it furniture?",
            'is_edible': "Is it edible?",
        }
        return questions.get(property_name, f"Is it {property_name}?")

    def _format_scale_question(self, property_name, threshold):
        """Format a scale property as a comparison question"""
        questions = {
            'size_scale_1_10': f"Is it larger than {threshold:.1f}/10 in size?",
            'weight_scale_1_10': f"Is it heavier than {threshold:.1f}/10 in weight?",
            'speed_scale_1_10': f"Is it faster than {threshold:.1f}/10 in speed?",
            'danger_to_humans_1_10': f"Is it more dangerous than {threshold:.1f}/10?",
            'fragility_scale_1_10': f"Is it more fragile than {threshold:.1f}/10?",
            'rarity_1_10': f"Is it rarer than {threshold:.1f}/10?",
            'typical_indoor_use_1_10': f"Is it more commonly found indoors than {threshold:.1f}/10?",
            'portability_1_10': f"Is it more portable than {threshold:.1f}/10?",
            'tech_complexity_1_10': f"Is it more technologically complex than {threshold:.1f}/10?",
            'edibleness': f"Is it more edible than {threshold:.1f}/10?",
        }
        return questions.get(property_name, f"Is {property_name} greater than {threshold:.1f}?")

    def filter_nouns(self, remaining_nouns, property_name, threshold, answer, target_noun=None):
        """
        Filter nouns based on the answer to a question
        """
        subset = self.kb[self.kb['noun'].isin(remaining_nouns)]

        if threshold is None:  # Binary property
            # Check missing data -> mask
            mask = (subset[property_name] == answer) | (subset[property_name].isna())
        else:  # Scale property
            if answer:
                mask = (subset[property_name] > threshold) | (subset[property_name].isna())
            else:
                mask = (subset[property_name] <= threshold) | (subset[property_name].isna())
                filtered = subset[subset[property_name] <= threshold]

        filtered = subset[mask] 
        new_remaining = set(filtered['noun'].values)
        eliminated_nouns = set(remaining_nouns) - new_remaining

        # Only warn if target noun is actually eliminated
        if target_noun is not None and target_noun in eliminated_nouns:
            print(f"⚠️ Target eliminated! {target_noun}: {property_name} = {self.kb[self.kb['noun'] == target_noun][property_name].values[0]}, answer = {answer}, threshold = {threshold}")

        return set(filtered['noun'].values)
        
    def ask_yes_no(self, prompt: str) -> Optional[bool]:
        """
        Ask user for yes/no. Returns True/False or None for 'quit'.
        Accepts: y/yes, n/no, q/quit
        """
        while True:
            ans = input(f"{prompt} (y/n, q to quit): ").strip().lower()
            if ans in ("y", "yes"):
                return True
            if ans in ("n", "no"):
                return False
            if ans in ("q", "quit", "exit"):
                return None
            print("Please answer 'y' or 'n' (or 'q' to quit).")

    def single_explination(self, noun, property_name, threshold, answer):
        """
        Explain why noun was eliminated by a question
        """
        row = self.kb[self.kb['noun'] == noun].iloc[0]
        
        if threshold is None:  # Binary
            actual_value = row[property_name]
            expected_value = answer
            explanation = f"  {noun}: {property_name} = {actual_value}, but answer was {expected_value}"
        else:  # Scale
            actual_value = row[property_name]
            expected_comparison = ">" if answer else "<="
            explanation = f"  {noun}: {property_name} = {actual_value:.1f}, but answer requires {property_name} {expected_comparison} {threshold:.1f}"
        return explanation
        
    def explain_elimination(self, eliminated_nouns, property_name, threshold, answer):
        """
        Explain why certain nouns were eliminated by a question
        Useful for debugging knowledge base inconsistencies
        """
        if len(eliminated_nouns) == 0:
            return []
        
        subset = self.kb[self.kb['noun'].isin(eliminated_nouns)]
        explanations = []
        
        for noun in list(eliminated_nouns)[:10]:  # Show first 10
            explanation = self.single_explination(noun, property_name, threshold, answer)
            explanations.append(explanation)
        
        if len(eliminated_nouns) > 10:
            explanations.append(f"  ... and {len(eliminated_nouns) - 10} more")
        
        return explanations

print("✅ TwentyQuestionsGame class defined")

✅ TwentyQuestionsGame class defined


In [21]:
# Cell 16 - PLAY GAME FUNCTION WITH BACKTRACKING
import random

def play_game_interactive(game, target_noun, max_questions=20, verbose=True, auto=True, 
                          enable_backtrack=True, debug_mode=False):
    """
    Play a game of 20 questions interactively with backtracking support

    Args:
        game: TwentyQuestionsGame instance
        target_noun: The noun to guess
        max_questions: Maximum questions allowed
        verbose: Print detailed output
        auto: Auto-answer from knowledge base (True) or manual input (False)
        enable_backtrack: Enable undo/correction functionality
        debug_mode: Show detailed elimination information

    Returns:
        dict with game results
    """
    if target_noun not in game.all_nouns:
        return {
            'success': False,
            'error': f"'{target_noun}' not in knowledge base"
        }

    remaining_nouns = game.all_nouns.copy()
    questions_asked = []
    answers_given = []
    properties_used = []  # Track which properties were used
    thresholds_used = []   # Track thresholds for scale questions
    
    # BACKTRACKING: Game state history
    game_history = []  # List of game states: (remaining_nouns, question_num, question, answer)
    
    def save_game_state(question_num, question_text, answer, prop, threshold):
        """Save current game state to history"""
        game_history.append({
            'question_num': question_num,
            'question': question_text,
            'answer': answer,
            'property': prop,
            'threshold': threshold,
            'remaining_nouns': remaining_nouns.copy(),
            'remaining_count': len(remaining_nouns)
        })
    
    def restore_game_state(state_index):
        """Restore game to a previous state"""
        if state_index < 0 or state_index >= len(game_history):
            return False
        
        # Restore to state before the specified question
        state = game_history[state_index]
        nonlocal remaining_nouns, questions_asked, answers_given, properties_used, thresholds_used
        
        remaining_nouns = state['remaining_nouns'].copy()
        questions_asked = questions_asked[:state['question_num'] - 1]
        answers_given = answers_given[:state['question_num'] - 1]
        properties_used = properties_used[:state['question_num'] - 1]
        thresholds_used = thresholds_used[:state['question_num'] - 1]
        
        # Remove all states after this one
        game_history[:] = game_history[:state_index + 1]
        return True
    
    def show_history():
        """Display game history"""
        print("\n" + "="*70)
        print("GAME HISTORY")
        print("="*70)
        for i, state in enumerate(game_history):
            print(f"\nState {i+1} (Q{state['question_num']}):")
            print(f"  Question: {state['question']}")
            print(f"  Answer: {state['answer']}")
            print(f"  Remaining nouns: {state['remaining_count']}")
        print("="*70)
    
    def ask_with_undo(prompt: str, question_num: int) -> Optional[bool]:
        """
        Ask user for yes/no with undo/correction options
        """
        while True:
            ans = input(f"{prompt}\n  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit): ").strip().lower()
            if ans in ("y", "yes"):
                return True
            if ans in ("n", "no"):
                return False
            if ans in ("u", "undo") and enable_backtrack:
                if len(game_history) > 0:
                    # To undo the last question, restore to the state before it
                    # game_history[-1] is after the last question, we want before it
                    if len(game_history) > 1:
                        prev_question = game_history[-2]['question_num']
                        print(f"\n🔄 UNDO: Going back to before question {game_history[-1]['question_num']}")
                        restore_game_state(len(game_history) - 2)
                        return ("UNDO", prev_question)
                    else:
                        # Only one question, go back to start
                        print(f"\n🔄 UNDO: Going back to beginning")
                        restore_game_state(0)
                        return ("UNDO", 0)
                else:
                    print("⚠️  Nothing to undo!")
                    continue
            if ans in ("h", "history") and enable_backtrack:
                show_history()
                continue
            if ans in ("d", "debug") and debug_mode:
                if len(game_history) > 0:
                    last_state = game_history[-1]
                    eliminated = game.all_nouns - last_state['remaining_nouns']
                    if len(eliminated) > 0:
                        print(f"\n🔍 DEBUG: Showing elimination details for last question:")
                        explanations = game.explain_elimination(
                            eliminated, 
                            last_state['property'], 
                            last_state['threshold'], 
                            last_state['answer']
                        )
                        for exp in explanations:
                            print(exp)
                continue
            if ans in ("q", "quit", "exit"):
                return None
            print("Please answer 'y', 'n', 'u' (undo), 'h' (history), 'd' (debug), or 'q' (quit).")

    if verbose:
        print("="*70)
        print(f"🎮 PLAYING 20 QUESTIONS")
        print(f"Target noun: {target_noun}")
        print("="*70)

    question_num = 0
    while question_num < max_questions:
        question_num += 1
        if verbose:
            print(f"\n--- Question {question_num} ---")
            print(f"Remaining possibilities: {len(remaining_nouns)}")

        # Check if we're down to one noun
        if len(remaining_nouns) == 1:
            final_guess = list(remaining_nouns)[0]
            questions_asked.append(f"Are you thinking of {final_guess}?")

            if verbose:
                print(f"\nQ{question_num}: Are you thinking of {final_guess}?")
                print(f"A: {'YES! ✅' if final_guess == target_noun else 'NO ❌'}")

            return {
                'success': final_guess == target_noun,
                'final_guess': final_guess,
                'target_noun': target_noun,
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given + [final_guess == target_noun],
                'remaining_nouns': 1,
                'game_history': game_history
            }

        # Last question must be a guess
        if question_num == max_questions:
            # Make best guess from remaining nouns
            final_guess = list(remaining_nouns)[0]  # Could be smarter here
            questions_asked.append(f"Are you thinking of {final_guess}?")

            if verbose:
                print(f"\nQ{question_num}: Are you thinking of {final_guess}? (Final guess)")
                print(f"A: {'YES! ✅' if final_guess == target_noun else 'NO ❌'}")

            return {
                'success': final_guess == target_noun,
                'final_guess': final_guess,
                'target_noun': target_noun,
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given + [final_guess == target_noun],
                'remaining_nouns': len(remaining_nouns),
                'game_history': game_history
            }
            
        # Find best question
        asked_questions = set()

        prop, threshold, question_text = game.find_best_question(remaining_nouns, asked_questions)
        
        if prop is None:
            # No good question found, make a guess
            final_guess = list(remaining_nouns)[0]
            questions_asked.append(f"Are you thinking of {final_guess}?")

            if verbose:
                print(f"\nQ{question_num}: Are you thinking of {final_guess}? (Forced guess)")
                print(f"A: {'YES! ✅' if final_guess == target_noun else 'NO ❌'}")

            return {
                'success': final_guess == target_noun,
                'final_guess': final_guess,
                'target_noun': target_noun,
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given + [final_guess == target_noun],
                'remaining_nouns': len(remaining_nouns),
                'game_history': game_history
            }

        # Record this question as asked
        asked_questions.add((prop, threshold))
        
        # Get the correct answer from knowledge base
        if auto:
            target_row = game.kb[game.kb['noun'] == target_noun].iloc[0]

            if threshold is None:  # Binary
                correct_answer = bool(target_row[prop])
            else:  # Scale
                correct_answer = float(target_row[prop]) > threshold
        else:
            if enable_backtrack:
                correct_answer = ask_with_undo(question_text, question_num)
            else:
                correct_answer = game.ask_yes_no(question_text)
            
            if isinstance(correct_answer, tuple) and correct_answer[0] == "UNDO":
                # User undid, adjust question number to continue from restored state
                _, restored_q = correct_answer
                question_num = restored_q
                continue
            
            if (correct_answer == None):
                return {
                    'success': False,
                    'error': 'User manually quit game.',
                    'questions_asked': question_num,
                    'questions': questions_asked,
                    'answers': answers_given,
                    'game_history': game_history
                }

        # Store state BEFORE filtering (so we can undo to this point)
        nouns_before = remaining_nouns.copy()
        
        questions_asked.append(question_text)
        answers_given.append(correct_answer)
        properties_used.append(prop)
        thresholds_used.append(threshold)

        if verbose:
            print(f"\nQ{question_num}: {question_text}")
            print(f"A: {'YES' if correct_answer else 'NO'}")

        # Filter remaining nouns
        remaining_nouns = game.filter_nouns(remaining_nouns, prop, threshold, correct_answer, target_noun=target_noun)
        
        # Debug: Show eliminated nouns
        if debug_mode:
            eliminated = nouns_before - remaining_nouns
            if len(eliminated) > 0:
                print(f"  🔍 Eliminated {len(eliminated)} nouns")
                if len(eliminated) <= 10:
                    explanations = game.explain_elimination(eliminated, prop, threshold, correct_answer)
                    for exp in explanations:
                        print(exp)
        
        # Save state AFTER filtering
        save_game_state(question_num, question_text, correct_answer, prop, threshold)

        if len(remaining_nouns) == 0:
            if verbose:
                print("\n❌ ERROR: No remaining nouns! Knowledge base inconsistency.")
            return {
                'success': False,
                'error': 'No remaining nouns',
                'questions_asked': question_num,
                'questions': questions_asked,
                'answers': answers_given,
                'game_history': game_history
            }

        # Prompt user if target noun is no longer in batch
        if debug_mode and target_noun not in remaining_nouns:
            print("⚠️ Warning ⚠️ Warning ⚠️ Warning ⚠️: Your selection eliminated your noun from the game!")
            last_state = game_history[-1]
            explanation = game.single_explination(
                target_noun,
                last_state['property'], 
                last_state['threshold'], 
                last_state['answer'])
            print(explanation + "\n")
            
    # Should not reach here
    return {
        'success': False,
        'error': 'Max questions reached',
        'questions_asked': max_questions,
        'remaining_nouns': len(remaining_nouns),
        'game_history': game_history
    }

# Initialize game
game = TwentyQuestionsGame(kb_df)

# Test with one noun
#test_noun = random.choice(list(game.all_nouns))
#print(f"\n\n🧪 Testing with: {test_noun}\n")
#result = play_game_interactive(game, test_noun, verbose=True)

#print(f"\n\n{'='*70}")
#print("GAME RESULT:")
#print(f"Success: {result['success']}")
#print(f"Questions used: {result.get('questions_asked', 'N/A')}")
#print("="*70)


Game initialized with 1852 nouns
Binary properties: 16
Scale properties: 9


# Backtracking Features

This notebook now includes backtracking capabilities to help debug knowledge base issues and handle user errors:

## Features:

1. **Undo Functionality**: Type 'u' or 'undo' during a question to go back one step
2. **History View**: Type 'h' or 'history' to see all questions and answers so far
3. **Debug Mode**: Type 'd' or 'debug' to see detailed elimination information
4. **Game History**: Full game state is saved after each question

## Usage Examples:

### Interactive Mode with Backtracking:
```python
result = play_game_interactive(game, "bicycle", verbose=True, auto=False, enable_backtrack=True)
```

### Debug Mode (shows eliminated nouns):
```python
result = play_game_interactive(game, "bicycle", verbose=True, auto=False, enable_backtrack=True, debug_mode=True)
```


In [13]:
# Example: Test backtracking with debug mode
# Uncomment to test

test_noun = "elephant"  # or any noun in your knowledge base
print(f"\n🧪 Testing backtracking with: {test_noun}\n")
result = play_game_interactive(
     game, 
     test_noun, 
     verbose=True, 
     auto=False,  # Manual input mode
     enable_backtrack=True,  # Enable undo/history
     debug_mode=True  # Show elimination details
)

print(f"\n\n{'='*70}")
print("GAME RESULT:")
print(f"Success: {result['success']}")
print(f"Questions used: {result.get('questions_asked', 'N/A')}")
print("="*70)



🧪 Testing backtracking with: elephant

🎮 PLAYING 20 QUESTIONS
Target noun: elephant

--- Question 1 ---
Remaining possibilities: 1852


Is it heavier than 2.0/10 in weight?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q1: Is it heavier than 2.0/10 in weight?
A: YES
  🔍 Eliminated 594 nouns

--- Question 2 ---
Remaining possibilities: 1258


Is it larger than 3.0/10 in size?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q2: Is it larger than 3.0/10 in size?
A: YES
  🔍 Eliminated 589 nouns

--- Question 3 ---
Remaining possibilities: 669


...


Q12: Is it more dangerous than 5.0/10?
A: NO
  🔍 Eliminated 3 nouns
  rhinoceros: danger_to_humans_1_10 = 9.0, but answer requires danger_to_humans_1_10 <= 5.0
  hippopotamus: danger_to_humans_1_10 = 8.0, but answer requires danger_to_humans_1_10 <= 5.0
  buffalo: danger_to_humans_1_10 = 7.0, but answer requires danger_to_humans_1_10 <= 5.0

--- Question 13 ---
Remaining possibilities: 3


Is it a physical object?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q13: Is it a physical object?
A: YES
  🔍 Eliminated 1 nouns
  african elephant: is_object = False, but answer was True

--- Question 14 ---
Remaining possibilities: 2


Is it larger than 9.0/10 in size?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q14: Is it larger than 9.0/10 in size?
A: YES
  🔍 Eliminated 1 nouns
  giraffe: size_scale_1_10 = 8.0, but answer requires size_scale_1_10 > 9.0

--- Question 15 ---
Remaining possibilities: 1

Q15: Are you thinking of elephant?
A: YES! ✅


GAME RESULT:
Success: True
Questions used: 15


In [6]:
# Test with one noun manually
test_noun = random.choice(list(game.all_nouns))
print(f"\n\n🧪 Testing with: {test_noun}\n")
result = play_game_interactive(game, 
     test_noun, 
     verbose=True, 
     auto=False,  # Manual input mode
     enable_backtrack=True,  # Enable undo/history
     debug_mode=True  # Show elimination details
)

print(f"\n\n{'='*70}")
print("GAME RESULT:")
print(f"Success: {result['success']}")
print(f"Questions used: {result.get('questions_asked', 'N/A')}")
print("="*70)



🧪 Testing with: banana plant

🎮 PLAYING 20 QUESTIONS
Target noun: banana plant

--- Question 1 ---
Remaining possibilities: 1852


Is it heavier than 2.0/10 in weight?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q1: Is it heavier than 2.0/10 in weight?
A: YES
  🔍 Eliminated 594 nouns

--- Question 2 ---
Remaining possibilities: 1258


Is it larger than 3.0/10 in size?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  n



Q2: Is it larger than 3.0/10 in size?
A: NO
  🔍 Eliminated 631 nouns

--- Question 3 ---
Remaining possibilities: 627


...


Q10: Is it an animal?
A: NO
  🔍 Eliminated 4 nouns
  alpaca: is_animal = True, but answer was False
  jynx: is_animal = True, but answer was False
  dolphin: is_animal = True, but answer was False
  falcon: is_animal = True, but answer was False

--- Question 11 ---
Remaining possibilities: 3


Is it a plant?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q11: Is it a plant?
A: YES
  🔍 Eliminated 1 nouns
  serpentine: is_plant = False, but answer was True

--- Question 12 ---
Remaining possibilities: 2


Is it edible?
  (y/n, 'u' to undo, 'h' for history, 'd' for debug, 'q' to quit):  y



Q12: Is it edible?
A: YES
  🔍 Eliminated 1 nouns
  hydrangea: is_edible = False, but answer was True

--- Question 13 ---
Remaining possibilities: 1

Q13: Are you thinking of banana plant?
A: YES! ✅


GAME RESULT:
Success: True
Questions used: 13


In [22]:
print("="*70)
print("RUNNING 5 TEST GAMES")
print("="*70)

# Manually enter the 5 nouns that you want to play
instructor_nouns = [
    "amazon",
    "bicycle",
    "computer",
    "apple",
    "ocean"
]

print(f"\nTest nouns: {instructor_nouns}\n")

# Verify all nouns are in knowledge base
missing_nouns = [n for n in instructor_nouns if n not in game.all_nouns]
if missing_nouns:
    print(f"⚠️  WARNING: These nouns are not in the knowledge base: {missing_nouns}")
    print("The game will fail for these nouns.\n")

# Run the games
results = []

for i, noun in enumerate(instructor_nouns, 1):
    if noun not in game.all_nouns:
        print(f"\n{'#'*70}")
        print(f"GAME {i}/5: {noun}")
        print(f"{'#'*70}")
        print(f"❌ ERROR: '{noun}' not found in knowledge base")
        results.append({
            'success': False,
            'target_noun': noun,
            'error': 'Not in knowledge base'
        })
        continue

    print(f"\n\n{'#'*70}")
    print(f"GAME {i}/5: {noun}")
    print(f"{'#'*70}")

    result = play_game_interactive(game, noun, verbose=True)
    results.append(result)

    print(f"\n✅ Game {i} complete")

print(f"\n\n{'='*70}")
print("ALL 5 GAMES COMPLETE")
print("="*70)

RUNNING 5 TEST GAMES

Test nouns: ['amazon', 'bicycle', 'computer', 'apple', 'ocean']



######################################################################
GAME 1/5: amazon
######################################################################
🎮 PLAYING 20 QUESTIONS
Target noun: amazon

--- Question 1 ---
Remaining possibilities: 1852

Q1: Is it heavier than 2.0/10 in weight?
A: YES

--- Question 2 ---
Remaining possibilities: 1258

Q2: Is it larger than 3.0/10 in size?
A: YES

--- Question 3 ---
Remaining possibilities: 669

Q3: Is it faster than 1.0/10 in speed?
A: NO

--- Question 4 ---
Remaining possibilities: 332

Q4: Is it more fragile than 2.0/10?
A: NO

--- Question 5 ---
Remaining possibilities: 175

Q5: Is it more portable than 1.0/10?
A: YES

--- Question 6 ---
Remaining possibilities: 112

Q6: Is it more commonly found indoors than 2.0/10?
A: NO

--- Question 7 ---
Remaining possibilities: 55

Q7: Is it a physical object?
A: YES

--- Question 8 ---
Remaining possibiliti

In [23]:
# Metrics
import pandas as pd, json, random

TEST_COUNT = 100
random.seed(42)

# Choose 10 nouns deterministically that are guaranteed to be in KB
all_nouns = list(game.all_nouns)
test_nouns = random.sample(all_nouns, min(TEST_COUNT, len(all_nouns)))
print("Testing nouns:", test_nouns)

results = []

for i, noun in enumerate(test_nouns, 1):
        print(f"\n{'#'*70}")
        print(f"GAME {i}/{len(test_nouns)}: {noun}")
        print(f"{'#'*70}")

        if noun not in game.all_nouns:
            print(f"❌ ERROR: '{noun}' not found in knowledge base")
            results.append({
                'target_noun': noun,
                'guessed': None,
                'questions': 0,
                'correct': False,
                'error': 'Not in knowledge base'
        })
            continue
    
        result = play_game_interactive(game, noun, verbose=False)

        # Get results
        results.append({
            'target': result.get('target_noun', noun),
            'guessed': result.get('final_guess'),
            'questions': result.get('questions_asked', 0),
            'correct': bool(result.get('success', False))
        })

# Store in dataframe
df = pd.DataFrame(results, columns=['target', 'guessed', 'questions', 'correct'])

display(df)

# Summary metrics 
accuracy = df['correct'].mean() * 100
average = df['questions'].mean()
print(f"\nSummary: Accuracy = {accuracy:.1f}% | Avg questions = {average:.1f} over {len(df)} tests.")
df[~df['correct']].to_csv('incorrect_guesses.csv', index=False)

df.to_csv("20Questions_performance_summary.csv", index=False)
game.kb.to_json("20Questions_knowledge_base.json", orient="records", indent=2)
  
print("Saved: 20Questions_performance_summary.csv and 20Questions_knowledge_base.json")

Testing nouns: ['lobster tail', 'jumping jack', 'urchin', 'mosquito net', 'lagoon', 'kite', 'curie', 'arcade machine', 'hakeem olajuwon', 'sunscreen', 'coin', 'wall sit', 'box jumps', 'malachite', 'cartoon', 'satellite', 'oscar robertson', 'rope ladder', 'halite', 'omaha', 'fossil', 'ice cream', 'camel', 'pickup truck', 'oak', 'microwave', 'desert', 'rainstorm', 'lemon', 'forest', 'environment', 'new orleans', 'dave debusschere', 'basket', 'tangerine', 'cable bicep curls', 'episode', 'jackson’s chameleon', 'tennis', 'hat', 'button', 'light', 'doctor', 'tailplane', 'reality show', 'dumbbell hammer curl', 'guinea pig', 'salmon', 'english springer spaniel', 'idaho', 'bullmastiff', 'boston terrier', 'puzzles', 'kyanite', 'revolution', 'dragonfruit', 'kabuto', 'orc', 'flamingo', 'ring', 'weedle', 'utah', 'ladybug', 'fern', 'chest press', 'sam jones', 'mouse', 'zircon', 'pressure washer', 'pike', 'gulf', 'kettlebell swings', 'education', 'glider', 'beagle', 'new mexico', 'italian wall lizard

,target,guessed,questions,correct
0,lobster tail,lobster tail,13,True
1,jumping jack,jumping jack,20,True
2,urchin,urchin,13,True
3,mosquito net,mosquito net,13,True
4,lagoon,lagoon,20,True
...,...,...,...,...
95,whiteboard,whiteboard,20,True
96,fireblade,fireblade,18,True
97,genealogy,genealogy,19,True
98,carbonatite,carbonatite,17,True



Summary: Accuracy = 87.0% | Avg questions = 16.2 over 100 tests.
Saved: 20Questions_performance_summary.csv and 20Questions_knowledge_base.json


## Report

### Overall Approach and Design Process
For this project, we first created a database structure of all the nouns and their property fields by determining boolean and numeric ratings. We focused our approach on property-based filtering so that the knowledge base would contain a set of attributes such as size, weight, danger level, etc. To build off that, we created binary and threshold-based scale questions to help balance categorical and numerical attributes. Furthermore, we needed to implement a strategy that involved using tie-breakers to prioritize certain properties as well as the process of highest elimination power for worst-case candidates (more so a greedy information gain approach). We also realized that some nouns were causing the model to eliminate potential guesses due to undefined property values and implemented a mask to prevent that. We implemented features such as backtracking to undo/redo a question, viewing history to view the recorded questions and answers played between the model and user, and debugging to view detailed elimination information. When playing the game, the user is also able to see a warning of whether the target noun has been eliminated due to their answer. The model needed iterative tuning in order to ask more meaningful questions as well as improve accuracy.

### Overall Performance
We evaluated and tested at least 10 random nouns to determine accuracy and the average number of questions asked until it has attempted to guess the target noun. With about 80-87% accuracy in guessing the target noun and the average number of questions ranging from 14 to 17 over 100 tests, the system manages to ask questions and guess the noun efficiently with some error. This could be due to the model facing trouble when it comes to learning property patterns or examining/narrowing possibilities to make an estimated final guess. Furthermore, the model also seems to struggle with abstract nouns due to a lack of detailed or conditional data. However, with the backtrack feature, changing the answer can cause the model to curate a more accurate next question that will lead a pathway to guessing the target noun.

### Performance Summary
The `20Questions_performance_summary.csv` table shows how many questions it took to guess the correct noun.
|   |target|guessed|questions|correct|
------|------|------|------|------|
1|lobster tail     |lobster tail     |13     |TRUE     |
2|jumping jack     |jumping jack     |20     |TRUE     |
3|urchin     |urchin     |13     |TRUE     |
4|mosquito net     |mosquito net     |13     |TRUE     |
5|lagoon     |lagoon     |20     |TRUE     |
6|kite     |kite     |14     |TRUE     |
8|curie     |curie     |13     |TRUE     |
9|arcade machine     |arcade machine     |20     |TRUE     |
10|hakeem olajuwon     |hakeem olajuwon     |14     |TRUE     |

All results are recorded in the `20Questions_performance_summary.csv` table and results may vary if ran this notebook locally due to nouns being generated by random for testing purposes.

### Future Work and Limitations
While accuracy is in the upper half, there is still room for improvement. A key observation when guessing nouns is to consider similar nouns. For example, while the target noun to guess is "elephant," it seems that there is confusion with the noun, "african elephant," since both nouns would be duplicates of each other and have very distinct differences between them that the system is unable to narrow down. We can see another example with the `incorrect_guesses.csv` table, guessing "nebraska" when the target noun "wisconsin". 
|target|guessed|questions|correct|
|------|------|------|------|
|oscar robertson     |elgin baylor     |20     |FALSE     |
|fossil     |tulsa     |20     |FALSE     |
|episode     |jason kidd     |20     |FALSE     |
|hat     |scarf     |20     |FALSE     |
|reality show     |documentary     |20     |FALSE     |
|boston terrier     |basset hound     |20     |FALSE     |
|kyanite     |tulsa     |20     |FALSE     |
|revolution     |ktmrc8     |20     |FALSE     |
|utah     |tulsa     |20     |FALSE     |
|sam jones     |ktmrc8     |20     |FALSE     |
|pressure washer     |helipad     |20     |FALSE     |
|glider     |homebrewing     |20     |FALSE     |
|wisconsin     |nebraska     |20     |FALSE     |

This is because one noun could be a subtype or regional version of another. When their numeric property values could differ by less than a certain threshold, it's causing both nouns to trigger the same path of yes/no questions. However, for these state nouns, their size scale, portability, and tech complexity are set to null for "nebraska," while "wisconsin" is set to numeric ratings for such properties. To address this, we can create a filter that prevents the system from completely eliminating the similar noun as long as the property difference does not exceed a set threshold. Merging similar nouns would increase question efficiency and accuracy, and avoid the overuse of elimination.